In [0]:
import requests, json

HEADERS = {"Authorization": f"DirectLogin token={'eyJhbGciOiJIUzI1NiJ9.eyIiOiIifQ.2d9bAa0bil_sU-DBCBg6ZLv9Gm7f7Mywnod7HkSXBsE'}"}


banks_resp = requests.get(f"{'https://apisandbox.openbankproject.com'}/obp/v5.1.0/banks", headers=HEADERS)
banks      = banks_resp.json().get("banks", [])

print(f"Banks available: {len(banks)}")
for b in banks[:5]:
    print(f"  Bank ID: {b.get('id'):<25} Name: {b.get('full_name')}")

In [0]:
import requests
import pandas as pd
from pyspark.sql import functions as F

HEADERS = {"Authorization": f"DirectLogin token={'eyJhbGciOiJIUzI1NiJ9.eyIiOiIifQ.2d9bAa0bil_sU-DBCBg6ZLv9Gm7f7Mywnod7HkSXBsE'}"}
BANK_ID = banks[0]["id"]   
OBP_BASE = "https://apisandbox.openbankproject.com"

def call_api(endpoint):
    """Safe API call with error handling."""
    url  = f"{OBP_BASE}/obp/v5.1.0{endpoint}"
    resp = requests.get(url, headers=HEADERS, timeout=30)
    if resp.status_code == 200:
        return resp.json()
    else:
        print(f"  ⚠ {endpoint} returned {resp.status_code}: {resp.text[:100]}")
        return None

def write_to_bronze(data, record_key, table_name, endpoint):
    """Flatten JSON and write to Bronze Delta table."""
    if data is None:
        return
    records = data.get(record_key, [])
    if not records:
        print(f"  ⚠ No records in key '{record_key}' for {table_name}")
        return
    df  = pd.json_normalize(records)
    sdf = spark.createDataFrame(df.astype(str))
    sdf = (sdf
           .withColumn("api_endpoint",   F.lit(endpoint))
           .withColumn("api_bank_id",    F.lit(BANK_ID))
           .withColumn("ingestion_time", F.current_timestamp()))
    (sdf.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"customer_360.bronze.{table_name}"))
    print(f"  ✓ {table_name:<35} {len(records):>6,} records")

print("  BRONZE LAYER — Open Bank Project API Ingest")

# 1. Banks
write_to_bronze(
    call_api("/banks"),
    "banks", "bronze_api_banks", "/banks"
)

# 2. Accounts (public)
write_to_bronze(
    call_api(f"/banks/{BANK_ID}/accounts/public"),
    "accounts", "bronze_api_accounts", f"/banks/{BANK_ID}/accounts/public"
)

# 3. Branches
write_to_bronze(
    call_api(f"/banks/{BANK_ID}/branches"),
    "branches", "bronze_api_branches", f"/banks/{BANK_ID}/branches"
)

# 4. ATMs
write_to_bronze(
    call_api(f"/banks/{BANK_ID}/atms"),
    "atms", "bronze_api_atms", f"/banks/{BANK_ID}/atms"
)

# 5. Products (bank products — loans, cards, accounts)
write_to_bronze(
    call_api(f"/banks/{BANK_ID}/products"),
    "products", "bronze_api_products", f"/banks/{BANK_ID}/products"
)

# 6. My accounts + transactions (authenticated)
write_to_bronze(
    call_api("/my/accounts"),
    "accounts", "bronze_api_my_accounts", "/my/accounts"
)

# 7. Transactions for first account found
my_accounts = call_api("/my/accounts")
if my_accounts and my_accounts.get("accounts"):
    first_account_id = my_accounts["accounts"][0]["id"]
    write_to_bronze(
        call_api(f"/my/banks/{BANK_ID}/accounts/{first_account_id}/transactions"),
        "transactions", "bronze_api_transactions",
        f"/my/banks/{BANK_ID}/accounts/{first_account_id}/transactions"
    )

print()


In [0]:
%sql
SELECT 'bronze_api_banks'        AS table_name, COUNT(*) AS rows FROM customer_360.bronze.bronze_api_banks        UNION ALL
SELECT 'bronze_api_accounts'     AS table_name, COUNT(*) AS rows FROM customer_360.bronze.bronze_api_accounts     UNION ALL
SELECT 'bronze_api_products'     AS table_name, COUNT(*) AS rows FROM customer_360.bronze.bronze_api_products 
ORDER BY table_name;

In [0]:
display(spark.table("customer_360.bronze.bronze_api_accounts"))